# GCSI Cross-Commodity Signal Assessment

## A method template for testing commodity sentiment against your own prices

This notebook is a **research template**, not a result. It shows a careful way to ask "does this sentiment
series line up with later returns?" across many commodities, drivers and settings without fooling yourself.
Bring your own prices. None are included, and nothing here reports performance.

### Pipeline

```
daily GCSI panel (from gcsi_bulk_history.ipynb)
  → signals built only from information available before each price date
  → forward log returns at several horizons, with UP / DOWN / BOTH regimes
  → correlation tests on the IN-SAMPLE period only, p-values adjusted for overlapping returns
  → Benjamini–Hochberg FDR across all in-sample tests
  → selected tests re-checked on a later HOLD-OUT period, separated by an embargo
  → per-commodity grade (A–F) from how many selected tests hold up
```

### Choices that keep the test honest

| Choice | Why |
|---|---|
| Signal for price date *P* uses headlines received up to 00:00 UTC on *P* | No look-ahead for any market closing on *P* |
| Selection on in-sample only; hold-out used once | Selecting on the full sample and then "validating" on part of it is circular |
| Embargo of the longest horizon between the two periods | Overlapping forward returns would otherwise leak across the split |
| Effective sample size `n / horizon` for p-values | Overlapping returns are autocorrelated; ordinary p-values would be far too small |
| FDR over every test run | With hundreds of tests, about 5% pass at p < 0.05 by chance alone |

### Caveats

- A correlation is not a trading strategy: no costs, sizing, capacity or execution are modelled.
- For the backfill period, the time each headline was received was reconstructed. Point-in-time accuracy there
  is approximate.
- History before the live period was scored after the fact by the current sentiment models, so results on it
  are in-sample for those models and are not independent evidence. Only data received live shows what a
  client would have seen at the time. Weight recent periods more heavily.

> **Disclaimer:** This notebook is provided for informational and research purposes only. Nothing here constitutes financial advice or a recommendation to buy, sell, or hold any asset. Sentiment scores are model outputs describing how news was judged at publication; they are not price forecasts.

In [ ]:
%pip install pandas numpy scipy matplotlib pyarrow

## Step 1: Configuration

`PRICES_CSV` must have the columns `date,ticker,close`, where `ticker` uses GCSI tickers (e.g. `BZ_COM`). Use a
**roll-adjusted continuous** futures series: raw front-month prices jump at each roll, and those jumps
contaminate returns.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

PANEL_DIR = Path('data/gcsi_daily')          # written by research/gcsi_bulk_history.ipynb
PRICES_CSV = Path('data/prices.csv')         # your prices: date,ticker,close
TICKERS = ['BZ_COM', 'CL_COM']
SAMPLE_START = '2016-01-01'                  # skip the thin early history

SENTIMENT_TYPE = 'short_term'
INDEX_TYPES = ['COMBINED', 'EXPLICIT']
LEVELS = ['all', 'driver']                   # add 'topic' for a finer (and much larger) sweep
WINDOWS = [7, 30, 90]                        # calendar days of news in each signal
MIN_LINKS = 20                               # fewer topic-links in a window → no signal that day
Z_LOOKBACK = 365                             # calendar days for the z-score baseline
HORIZONS = {'1w': 5, '1m': 21, '3m': 63}     # forward return horizons in trading days
REGIME_LOOKBACK = 63
REGIMES = ['BOTH', 'UP', 'DOWN']
IN_SAMPLE_FRACTION = 0.6
ALPHA = 0.05

EMBARGO = max(HORIZONS.values())
print(f'Tickers: {TICKERS} | {SENTIMENT_TYPE} | levels {LEVELS} | windows {WINDOWS} | horizons {list(HORIZONS)}')

## Step 2: Prices, forward returns and regimes

In [ ]:
prices = pd.read_csv(PRICES_CSV, parse_dates=['date'])
prices = prices[prices['ticker'].isin(TICKERS) & (prices['date'] >= SAMPLE_START)]


def returns_frame(ticker: str) -> pd.DataFrame:
    close = prices[prices['ticker'] == ticker].set_index('date')['close'].sort_index()
    if (close <= 0).any():
        # Back-adjusted series (and WTI in April 2020) can go negative; log returns need positive prices.
        print(f'{ticker}: {(close <= 0).sum()} non-positive prices dropped; consider a ratio-adjusted series')
        close = close[close > 0]
    frame = pd.DataFrame({'close': close})
    for label, h in HORIZONS.items():
        frame[f'ret_{label}'] = np.log(close.shift(-h) / close)
    trailing = np.log(close / close.shift(REGIME_LOOKBACK))
    frame['regime'] = np.where(trailing >= 0, 'UP', 'DOWN')
    frame.loc[trailing.isna(), 'regime'] = None
    return frame


returns = {t: returns_frame(t) for t in TICKERS if (prices['ticker'] == t).any()}
for ticker, frame in returns.items():
    print(f"{ticker}: {len(frame):,} trading days {frame.index.min():%Y-%m-%d} → {frame.index.max():%Y-%m-%d}")
missing = set(TICKERS) - set(returns)
if missing:
    print('No prices for:', sorted(missing))

## Step 3: Signals with no look-ahead

For each `(index_type, level, name)` series, daily sums go onto a full calendar, with zero where there was no
news. That is correct for sums. Three signal families follow:

| Metric | Definition |
|---|---|
| `avg` | Weighted average over the window: `Σ sentiment_sum / Σ headline_count` |
| `avg_z` | `avg` as a z-score against its own trailing `Z_LOOKBACK`-day history |
| `volume_z` | `log(Σ headline_count)` as a z-score: attention, not direction |

The signal is then **shifted one calendar day**, so the value used on price date *P* contains only headlines
received before 00:00 UTC on *P*.

In [ ]:
def build_signals(ticker: str) -> pd.DataFrame:
    panel = pd.read_parquet(PANEL_DIR / f'{ticker}_daily.parquet')
    panel = panel[(panel['sentiment_type'] == SENTIMENT_TYPE) & panel['index_type'].isin(INDEX_TYPES)
                  & panel['level'].isin(LEVELS) & (panel['date'] >= pd.Timestamp(SAMPLE_START) - pd.Timedelta(days=Z_LOOKBACK))]
    calendar = pd.date_range(panel['date'].min(), returns[ticker].index.max(), freq='D')
    out = {}
    for (index_type, level, name), group in panel.groupby(['index_type', 'level', 'name']):
        daily = group.set_index('date')[['sentiment_sum', 'headline_count']].reindex(calendar, fill_value=0.0)
        for window in WINDOWS:
            rolled = daily.rolling(window).sum()
            avg = (rolled['sentiment_sum'] / rolled['headline_count']).where(rolled['headline_count'] >= MIN_LINKS)
            volume = np.log1p(rolled['headline_count'])
            base = avg.rolling(Z_LOOKBACK, min_periods=Z_LOOKBACK // 2)
            vbase = volume.rolling(Z_LOOKBACK, min_periods=Z_LOOKBACK // 2)
            for metric, series in {
                'avg': avg,
                'avg_z': (avg - base.mean()) / base.std(),
                'volume_z': (volume - vbase.mean()) / vbase.std(),
            }.items():
                out[(index_type, level, name, metric, window)] = series.shift(1)   # known before price date
    signals = pd.DataFrame(out)
    signals.columns.names = ['index_type', 'level', 'name', 'metric', 'window']
    return signals.reindex(returns[ticker].index)


signals = {}
for ticker in returns:
    signals[ticker] = build_signals(ticker)
    print(f'{ticker}: {signals[ticker].shape[1]} signal series')

## Step 4: In-sample tests with overlap-adjusted p-values

The in-sample period is the first `IN_SAMPLE_FRACTION` of each ticker's dates. Every
(signal × horizon × regime) pair gets a Pearson correlation. The t-statistic uses `n_eff = n / horizon`, because
consecutive h-day forward returns share h − 1 days.

In [ ]:
def corr_test(x: pd.Series, y: pd.Series, horizon: int) -> tuple[float, int, float]:
    mask = x.notna() & y.notna()
    n = int(mask.sum())
    n_eff = n / horizon
    if n_eff < 20:
        return np.nan, n, np.nan
    r = float(np.corrcoef(x[mask], y[mask])[0, 1])
    t = r * np.sqrt((n_eff - 2) / max(1e-12, 1 - r * r))
    return r, n, float(2 * stats.t.sf(abs(t), df=n_eff - 2))


def split(ticker: str):
    dates = returns[ticker].index
    cut = int(len(dates) * IN_SAMPLE_FRACTION)
    return dates[:cut], dates[cut + EMBARGO:]


def run_tests(ticker: str, dates: pd.DatetimeIndex, columns=None) -> pd.DataFrame:
    frame, sig = returns[ticker].loc[dates], signals[ticker].loc[dates]
    rows = []
    for column in (columns if columns is not None else sig.columns):
        x = sig[column]
        for label, h in HORIZONS.items():
            for regime in REGIMES:
                mask = frame['regime'].notna() if regime == 'BOTH' else frame['regime'] == regime
                r, n, p = corr_test(x[mask], frame.loc[mask, f'ret_{label}'], h)
                rows.append((*column, label, regime, r, n, p))
    return pd.DataFrame(rows, columns=[*signals[ticker].columns.names, 'horizon', 'regime', 'r', 'n', 'p'])


def bh_fdr(p: pd.Series, alpha: float = ALPHA) -> pd.Series:
    valid = p.dropna().sort_values()
    m = len(valid)
    passed = valid <= alpha * np.arange(1, m + 1) / m
    cutoff = valid[passed].max() if passed.any() else -1
    return (p <= cutoff) & p.notna()


in_sample = {}
for ticker in returns:
    is_dates, ho_dates = split(ticker)
    tests = run_tests(ticker, is_dates)
    tests['selected'] = bh_fdr(tests['p'])
    in_sample[ticker] = tests
    print(f"{ticker}: in-sample {is_dates[0]:%Y-%m-%d} → {is_dates[-1]:%Y-%m-%d} | {tests['p'].notna().sum():,} tests | "
          f"{tests['selected'].sum()} pass BH-FDR at {ALPHA} | hold-out from {ho_dates[0]:%Y-%m-%d}")

## Step 5: Hold-out check

Only the tests selected in-sample are re-run on the hold-out period. A test **holds up** when the hold-out
correlation has the same sign and p < `ALPHA`, both judged with the same overlap-adjusted test.

In [ ]:
keys = ['index_type', 'level', 'name', 'metric', 'window', 'horizon', 'regime']
held = {}
for ticker, tests in in_sample.items():
    chosen = tests[tests['selected']]
    if chosen.empty:
        held[ticker] = chosen.assign(r_hold=np.nan, p_hold=np.nan, holds=False)
        continue
    _, ho_dates = split(ticker)
    columns = list(dict.fromkeys(tuple(row) for row in chosen[keys[:5]].itertuples(index=False)))
    hold = run_tests(ticker, ho_dates, columns).rename(columns={'r': 'r_hold', 'n': 'n_hold', 'p': 'p_hold'})
    merged = chosen.merge(hold, on=keys, how='left')
    merged['holds'] = (np.sign(merged['r']) == np.sign(merged['r_hold'])) & (merged['p_hold'] < ALPHA)
    held[ticker] = merged
    print(f"{ticker}: {len(merged)} selected → {int(merged['holds'].sum())} hold up on the hold-out")

## Step 6: Cross-commodity summary

The grade cut-offs below are illustrative. Set your own. In particular, a single test that holds up can still be
luck.

| Grade | Rule |
|---|---|
| A | ≥ 5 tests hold up, across ≥ 2 horizons |
| B | ≥ 3 tests hold up |
| C | 1–2 tests hold up |
| D | Tests passed in-sample, but none held up |
| F | Nothing passed in-sample |

In [ ]:
def grade(frame: pd.DataFrame) -> str:
    ok = frame[frame['holds']] if not frame.empty else frame
    if frame.empty:
        return 'F'
    if len(ok) >= 5 and ok['horizon'].nunique() >= 2:
        return 'A'
    return 'B' if len(ok) >= 3 else 'C' if len(ok) >= 1 else 'D'


summary = pd.DataFrame([
    {
        'ticker': ticker,
        'tests_run': int(in_sample[ticker]['p'].notna().sum()),
        'selected_in_sample': len(frame),
        'hold_up': int(frame['holds'].sum()) if not frame.empty else 0,
        'median_abs_r_hold': frame.loc[frame['holds'], 'r_hold'].abs().median() if not frame.empty else np.nan,
        'grade': grade(frame),
    }
    for ticker, frame in held.items()
])
display(summary.round(3))

survivors = pd.concat([f[f['holds']].assign(ticker=t) for t, f in held.items() if not f.empty], ignore_index=True) \
    if any(not f.empty for f in held.values()) else pd.DataFrame()
if not survivors.empty:
    display(survivors.sort_values('p_hold')[['ticker', *keys, 'r', 'r_hold', 'p_hold', 'n_hold']].head(20).round(4))

## Step 7: How much did selection decay?

In-sample correlation against hold-out correlation for every selected test. Points near the diagonal held up.
Points near zero, or across the axis, did not. Some decay is normal; selection always inflates in-sample values.

In [ ]:
selected = pd.concat([f.assign(ticker=t) for t, f in held.items() if not f.empty], ignore_index=True) \
    if any(not f.empty for f in held.values()) else pd.DataFrame()
if selected.empty:
    print('No in-sample selections to compare.')
else:
    fig, ax = plt.subplots(figsize=(6.5, 6))
    for ticker, group in selected.groupby('ticker'):
        ax.scatter(group['r'], group['r_hold'], s=18, alpha=0.6, label=ticker)
    lim = max(selected['r'].abs().max(), selected['r_hold'].abs().max()) * 1.1
    ax.plot([-lim, lim], [-lim, lim], color='grey', lw=0.8, ls='--')
    ax.axhline(0, color='grey', lw=0.6)
    ax.axvline(0, color='grey', lw=0.6)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_xlabel('In-sample r')
    ax.set_ylabel('Hold-out r')
    ax.set_title('Selected tests: in-sample vs hold-out correlation')
    ax.legend()
    plt.tight_layout()
    plt.show()